In [81]:
%pip install requests pandas python-dotenv tiktoken tqdm

Note: you may need to restart the kernel to use updated packages.


In [82]:
import requests
import pandas as pd
import os
import base64
from dotenv import load_dotenv
import json
import tiktoken
from tqdm import tqdm

load_dotenv()

True

In [83]:
ANALYZE_EXTENSIONS = ['php', 'tsx', 'ts', 'js', 'jsx', 'html', 'java', 'go', 'py', 'rb', 'c']
MAX_VULNERABILITY_FILES = 15 #threshold for number of files to analyze -> aims to avoid commits with too many files
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

In [84]:
def load_existing_data(file_name):
    try:
        return pd.read_csv(file_name)
    except FileNotFoundError:
        return pd.DataFrame()

# Function to count tokens in file content
def count_tokens(file_content, from_base64=False):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    if from_base64:
        file_content = baseToString(file_content)
    tokens = tokenizer.encode(file_content)
    return len(tokens)

def baseToString(encoded_data):
    decoded_data = base64.b64decode(encoded_data)
    decoded_string = decoded_data.decode('utf-8')
    return decoded_string

# Function to get file content from GitHub
def getFileContent(repo, path, ref, raw=False):
    url = f"https://api.github.com/repos/{repo}/contents/{path}?ref={ref}"
    headers = {'Authorization': f'token {GITHUB_TOKEN}'}
    if raw:
        headers['Accept'] = 'application/vnd.github.raw+json'
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.text if raw else baseToString(response.json()["content"])
    return None

# Function to process each vulnerability
def process_vulnerability(row, unique_id_start, existing_keys):
    patches = json.loads(row['files'])
    commit_id = row['commit']
    processed_files = []

    if len(patches) > MAX_VULNERABILITY_FILES:
        return processed_files, unique_id_start
    for patch in patches:
        filename = patch["filename"]
        ending = filename.split(".")[-1]
        if ending not in ANALYZE_EXTENSIONS:
            continue
        if file_exists(existing_keys, (row['vulnerability_id'], filename)):
            continue
        try:
            new_file = getFileContent(row['repo'], filename, commit_id, raw=True)
            old_file = getFileContent(row['repo'], filename, f"{commit_id}^", raw=True) if "status" in patch and patch["status"] == "modified" else None
            old_file_tokens = count_tokens(old_file) if old_file else None
            processed_files.append({
                'file_id': unique_id_start,
                'vulnerability_id': row['vulnerability_id'],
                "CWE_ID": row["cwe_id"],
                "CVE_ID": row["cve_id"],
                'filename': filename,
                'file_before': old_file,
                'file_after': new_file,
                'patch': patch.get("patch"),
                'file_tokens': int(old_file_tokens),
            })
            unique_id_start += 1
        except Exception as e:
            print(f"Error processing file {filename} in commit {commit_id}: {str(e)}")
    return processed_files, unique_id_start

def file_exists(existing_keys, key):
    return key in existing_keys

# Fetch proper github file content

In [85]:
# Load vulnerabilities data from CSV
input_csv = 'vulnerabilities.csv'
vulnerabilities = load_existing_data(input_csv)

# Load existing files data to determine the starting file_id
existing_files_csv = 'files.csv'
existing_files = load_existing_data(existing_files_csv)
if not existing_files.empty:
    last_file_id = existing_files['file_id'].max()
    unique_id_start = last_file_id + 1
    existing_keys = set(existing_files[['vulnerability_id', 'filename']].apply(tuple, axis=1))
else:
    unique_id_start = 1
    existing_keys = set()

# Process vulnerabilities and save results
all_processed_files = []
for index, row in tqdm(vulnerabilities.iterrows(), total=vulnerabilities.shape[0], desc="Processing vulnerabilities"):
    processed_files, unique_id_start = process_vulnerability(row, unique_id_start, existing_keys)
    all_processed_files.extend(processed_files)
# Convert to DataFrame and save to CSV
output_df = pd.DataFrame(all_processed_files)

# If the files.csv already exists, append to it; otherwise, create it
if not existing_files.empty:
    combined_df = pd.concat([existing_files, output_df], ignore_index=True)
else:
    combined_df = output_df

combined_df.to_csv(existing_files_csv, index=False)

Processing vulnerabilities: 100%|██████████| 8/8 [00:00<00:00, 2666.65it/s]


In [87]:
combined_df.head()

,file_id,vulnerability_id,CWE_ID,CVE_ID,filename,file_before,file_after,patch,file_tokens
0,1,1,CWE-79,CVE-2015-10128,js/jquery.prettyPhoto.js,/* -------------------------------------------...,/* -------------------------------------------...,"@@ -2,6 +2,910 @@\n \tClass: prettyPhoto\n \tU...",5832.0
1,2,1,CWE-79,CVE-2015-10128,rt-prettyphoto.php,<?php\r\n/*\r\nPlugin Name: Royal PrettyPhoto\...,<?php\r\n/*\r\nPlugin Name: Royal PrettyPhoto\...,"@@ -5,7 +5,7 @@\n Description: This plugin wil...",5259.0
2,3,2,CWE-79,CVE-2017-20188,WebRoot/js/ajax/dwt/xforms/XFormItem.js,/*\n * ***** BEGIN LICENSE BLOCK *****\n * Zim...,/*\n * ***** BEGIN LICENSE BLOCK *****\n * Zim...,"@@ -688,7 +688,7 @@ XFormItem.prototype.setErr...",42055.0
3,4,3,CWE-79,CVE-2018-25097,ds-compositionengine/src/main/java/org/acumos/...,/*-\n * ===============LICENSE_START==========...,/*-\n * ===============LICENSE_START==========...,"@@ -25,6 +25,7 @@\n import org.acumos.designst...",1082.0
4,5,3,CWE-79,CVE-2018-25097,ds-compositionengine/src/main/java/org/acumos/...,/*-\n * ===============LICENSE_START==========...,/*-\n * ===============LICENSE_START==========...,"@@ -33,6 +33,7 @@\n import org.acumos.designst...",7243.0


# Create datasets for single CWE